# Integer linear programming with native FlagQuantum synthesis

Maximize c*x with x0,x1,x2 in {0,1,2,3} and all three original inequalities A*x<=b. Preserve three layers, CVaR 0.7 and a 90-iteration setting. The two slack registers are modeled outputs, not clean workspace.


In [ ]:
"""Complete native integer optimization with explicit numeric and slack ports."""

import matplotlib.pyplot as plt

import flagquantum as fq
from examples.synthesis.applications.optimization.integer_linear_programming.overview.plots import (
    optimization_figure,
)
from examples.synthesis.applications.optimization.integer_linear_programming.overview.workflow import (
    NAMES,
    SOURCE_A,
    SOURCE_B,
    SOURCE_C,
    model_row,
    run_workflow,
)
from examples.synthesis.graph_optimization import INITIAL_PARAMETERS
from flagquantum import synth
from flagquantum.synth import (
    RX,
    Output,
    QBit,
    QNum,
    allocate,
    hadamard_transform,
    phase,
    qfunc,
    suzuki_trotter,
)

MIXER = fq.X(0) + fq.X(1)


@qfunc
def main(
    params: list[float],
    x0: Output[QNum],
    x1: Output[QNum],
    x2: Output[QNum],
    s0: Output[QBit],
    s1: Output[QBit],
) -> None:
    allocate(2, x0)
    allocate(2, x1)
    allocate(2, x2)
    allocate(s0)
    allocate(s1)
    hadamard_transform(x0)
    hadamard_transform(x1)
    hadamard_transform(x2)
    hadamard_transform(s0)
    hadamard_transform(s1)
    for layer in range(3):
        phase(
            -(
                10 * x0 * x1
                + 10 * x0 * x2
                - x0
                + 10 * x1 * x2
                - 2 * x1
                - 3 * x2
                + 10 * (s0 + x0 + x1 + x2 - 1.0) ** 2
                + 10 * (s1 + x0 + x1 + x2 - 1.0) ** 2
            ),
            params[layer],
        )
        suzuki_trotter(MIXER, params[3 + layer] / 2, 1, 1, x0)
        suzuki_trotter(MIXER, params[3 + layer] / 2, 1, 1, x1)
        suzuki_trotter(MIXER, params[3 + layer] / 2, 1, 1, x2)
        RX(params[3 + layer], s0)
        RX(params[3 + layer], s1)


## Synthesize and read the full named joint domain

The native quantum function above retains the frozen quantum encoding. Its penalty cost is reported separately from the original objective and feasibility. Angles are radians. No physical wires or manual bitstring decoding are required.


In [ ]:
print("Original model", SOURCE_A, SOURCE_B, SOURCE_C)
initial_program = synth.synthesize(
    main,
    classical_inputs={"params": INITIAL_PARAMETERS},
    max_qubits=8,
    max_workspace=0,
    max_operations=5000,
    memory_limit_bytes=4 * 1024**2,
)
initial_readout = dict(
    initial_program.probabilities(fq.run(initial_program), registers=NAMES)
)
print("Joint assignments", len(initial_readout))
print("Resources", initial_program.report["resources"])
print("Temporary workspace", initial_program.report["workspace_qubits"])


## Actual bounded optimization and observations

Public native Nelder-Mead with step 0.07 replaces the source private optimizer. Exact local probabilities replace shot estimation during CVaR search. Every evaluation synthesizes and executes. Final observations use 1000 actual CPU samples each, seeds 2718 optimized and 31415 uniform. Report convergence and evaluation counts; do not infer success from the iteration budget.


In [ ]:
workflow = run_workflow()
print("Initial CVaR", workflow.trace[0].cost)
print("Final CVaR", workflow.optimizer.value)
print("Iterations", workflow.optimizer.iterations)
print("Actual evaluations", workflow.optimizer.evaluations)
print("Converged", workflow.optimizer.converged)
for sample in workflow.samples[:5]:
    print(sample)
print(
    "Feasible optimized observations",
    sum(row.count for row in workflow.samples if row.feasible),
)
print(
    "Feasible uniform observations",
    sum(row.count for row in workflow.uniform if row.feasible),
)


## Observed feasible assignment and source figure roles

Keep all three inequalities explicit, including their redundant rows. Rank feasible observations by the original maximization objective. If none is observed, report None. Keep the trace and common 30-bin histogram, and show the original integer coordinates. Both slack outputs remain part of the quantum joint distribution.


In [ ]:
best = workflow.best_feasible
print("Best observed feasible", best)
if best is not None:
    print("Original assignment", best.assignment[:3])
    print("All constraint left sides", best.lhs)
    print("Original objective", best.objective)
    print("Modeled slack outputs", best.assignment[3:])
figure = optimization_figure(workflow)
plt.show()
plt.close(figure)


## Independent bounded reference after sampling

Enumerating 64 original-domain assignments replaces Couenne after the optimization and observations. It never supplies parameters, probabilities or missing observations. A finite sampled optimum does not prove optimizer convergence, hardware parity or quantum advantage.


In [ ]:
exact = model_row((*workflow.exact_reference, 0, 0))
print("Exact bounded original-model reference", workflow.exact_reference)
print("Reference objective", exact.objective)
print("Reference constraint left sides", exact.lhs)
print("Actual best feasible observation", workflow.best_feasible)
